# TC_10: CNN clean baseline
Full diagnostic pipeline on clean image data using simple CNN model.
UQ: Deep Ensemble, MC Dropout
XAI: Grad-CAM

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from utils.visualization import plot_gradcam_sample

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_image_data
from src.data_diagnostics.quality_checks import check_image_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.image_models import train_simple_cnn, evaluate_image_model
from src.inference_diagnostics.uncertainty import mc_dropout_prediction_img, deep_ensemble_prediction_img
from src.inference_diagnostics.explainability import gradcam_img, collect_gradcam_feedback
from src.inference_diagnostics.flagging import assign_flags, evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report


config = load_config()
tracker = PerformanceTracker()

### Load image data and EDA


In [2]:
train_set, test_set = load_image_data(config)
report = check_image_quality(train_set, config)
plot_class_distribution(report, "CNN CIFAR-10 Baseline", config)

Loading dataset.
Loaded: 50000 train and 10000 test
Image size: 32
Classes: 10
EDA started for image data.
Total images: 50000
Classes: 10
Distribution: {0: 5000, 1: 5000, 2: 5000, 3: 5000, 4: 5000, 5: 5000, 6: 5000, 7: 5000, 8: 5000, 9: 5000}
Imbalance ratio: 1.0
EDA completed for CIFAR-10
Saved: results/figures\class_distribution_cnn_cifar-10_baseline.png


### Train CNN baseline

In [3]:
tracker.start_performance_track()
cnn_model = train_simple_cnn(train_set, config)
tracker.stop_performance_track("CNN training")

tracker.start_performance_track()
cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
tracker.stop_performance_track("CNN evaluation")

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training: Time:529.67s, Memory:699.69MB
 Image model evaluation started.
{'0': {'precision': 0.8023483365949119, 'recall': 0.82, 'f1-score': 0.811078140454995, 'support': 1000.0}, '1': {'precision': 0.8905775075987842, 'recall': 0.879, 'f1-score': 0.8847508807247106, 'support': 1000.0}, '2': {'precision': 0.6829268292682927, 'recall': 0.7, 'f1-score': 0.691358024691358, 'support': 1000.0}, '3': {'precision': 0.6108829568788501, 'recall': 0.595, 'f1-score': 0.6028368794326241, 'support': 1000.0}, '4': {'precision': 0.7037037037037037, 'recall': 0.779, 'f1-score': 0.7394399620313241, 'support': 1000.0}, '5': {'precision': 0.7209567198177677, 'recall': 0.633, 'f1-score': 0.6741214057507987, 'support': 1000.0}, '6': {'precision': 0.7967244701348748, 'recall': 0.827, 'f1-score': 0.8115799803729146, 'support': 1000.0}, '7': {'precision': 0.81855249745158, 'recall': 0.803, 'f1-score'

{'operation': 'CNN evaluation', 'time_seconds': 4.65, 'memory_usage': 9.84}

### Uncertainty Quantification (MC Dropout + Deep Ensemble)

In [4]:
# MC Dropout
tracker.start_performance_track()
mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
tracker.stop_performance_track("CNN MC Dropout")

mc_threshold = round(float(np.percentile(mc_uncertainty, 90)), 4)
print(f"MC Dropout uncertainty status:")
print(f"Mean: {mc_uncertainty.mean()}")
print(f"Max: {mc_uncertainty.max()}")
print(f" 90th percentile (threshold): {mc_threshold}")

plot_uncertainty_distribution(mc_uncertainty, "CNN MC Dropout", mc_threshold, config)

MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for images.
CNN MC Dropout: Time:220.84s, Memory:19.36MB
MC Dropout uncertainty status:
Mean: 0.028895556926727295
Max: 0.17375586926937103
 90th percentile (threshold): 0.0804
Saved: results/figures\uncertainty_distribution_cnn_mc_dropout.png


In [5]:
# Deep Ensemble
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set  )
tracker.stop_performance_track("CNN Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "CNN Deep Ensemble", de_threshold, config)

Deep Ensemble started for images.
Training model with seed 0
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 1
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 2
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 3
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 4
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Deep Ensemble finished for images.
CNN Deep Ensemble prediction: Time:2508.89s, Memory:1.79MB
Deep Ensembl uncertainty status:
Mean: 0.04080768674612045
Max: 0.18797513842582703
 90th percentile (threshold): 0.1051
Saved: results/figures\uncertainty_distribution_cnn_deep_ensemble.png


### Explainability (Grad-CAM and Human feedback)

In [6]:
tracker.start_performance_track()
heatmaps = gradcam_img(cnn_model, test_set, config)
tracker.stop_performance_track("CNN Grad-CAM")

# Show grid for expert review
plot_gradcam_sample(test_set, heatmaps, 20, config, "CNN CIFAR-10 Baseline", save = True)


GradCam started.
GradCam finished. Explained: 20 samples.
CNN Grad-CAM: Time:0.11s, Memory:15.69MB
Saved: results/figures\gradcam_sample_cnn_cifar-10_baseline.png


### Collect human feedback

In [7]:
consistency_scores, correct, incorrect, partial = collect_gradcam_feedback(20)
print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")

Review the 20 heatmap above
Enter verdicts as comma separated numbers. Ex:1,2,3,2,2,1
1 = Correct (model is looking at the correct areas)
2 = Partial (model is looking at some correct areas)
3 = Incorrect (model is looking at wrong areas)
Correct: 0, Incorrect: 12, Partial: 8
Human feedback: correct: 0, incorrect: 12, partial: 8


### Flagging (UQ + Grad-CAM feedback)

In [8]:
# Get true  labels for first 20 samples
y_true_20 = np.array([test_set[i][1] for i in range(20)])

# MC Dropout
mc_y_pred_20 = mc_means_probabilities[:20].argmax(axis = 1)
mc_flags_reviewed = assign_flags(mc_uncertainty[:20], consistency_scores, mc_threshold, 0.5)
mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_20, y_true_20)
plot_flag_distribution(mc_flags_reviewed, "CNN MC + Grad-CAM Reviewed", config)

# Deep Ensemble
de_y_pred_20 = de_means_probabilities[:20].argmax(axis = 1)
de_flags_reviewed = assign_flags(de_uncertainty[:20], consistency_scores, de_threshold, 0.5)
de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_20, y_true_20)
plot_flag_distribution(de_flags_reviewed, "CNN DE + Grad-CAM Reviewed", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 13 Accuracy:69.2%
GREEN: Count: 7 Accuracy:85.7%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_+_grad-cam_reviewed.png
RED: Count: 1 Accuracy:100.0%
YELLOW: Count: 12 Accuracy:66.7%
GREEN: Count: 7 Accuracy:100.0%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_cnn_de_+_grad-cam_reviewed.png


### Full flagging with constant consistency for all sample

In [9]:
# Full test set flagging (UQ only, no human review)
fully_y_true = np.array([test_set[i][1] for i in range(len(test_set))])
fake_consistency = np.ones(len(mc_uncertainty))

# MC only
full_mc_y_prediction = mc_means_probabilities.argmax(axis = 1)
mc_flags_full = assign_flags(mc_uncertainty, fake_consistency, mc_threshold, 0.5)
mc_full_result = evaluate_flags(mc_flags_full, full_mc_y_prediction, fully_y_true)
plot_flag_distribution(mc_flags_full, "CNN MC UQ Only Full", config)

# Deep Ensemble only
full_de_y_prediction = de_means_probabilities.argmax(axis = 1)
de_flags_full = assign_flags(de_uncertainty, fake_consistency, de_threshold, 0.5)
de_full_result = evaluate_flags(de_flags_full, full_de_y_prediction, fully_y_true)
plot_flag_distribution(de_flags_full, "CNN DE UQ Only Full", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 999 Accuracy:37.4%
GREEN: Count: 9001 Accuracy:81.5%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_uq_only_full.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 1001 Accuracy:44.7%
GREEN: Count: 8999 Accuracy:86.0%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_uq_only_full.png


In [10]:
# 20 sample test set flagging (UQ only, no human review)
fake_consistency_20 = np.ones(20)

# MC only
mc_flags_20 = assign_flags(mc_uncertainty[:20], fake_consistency_20, mc_threshold, 0.5)
mc_20_result = evaluate_flags(mc_flags_20, mc_y_pred_20, y_true_20)
plot_flag_distribution(mc_flags_20, "CNN MC UQ Only for 20", config)

# Deep Ensemble only
de_flags_20 = assign_flags(de_uncertainty[:20], fake_consistency_20, de_threshold, 0.5)
de_20_result = evaluate_flags(de_flags_20, de_y_pred_20, y_true_20)
plot_flag_distribution(de_flags_20, "CNN DE UQ Only for 20", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 1 Accuracy:0.0%
GREEN: Count: 19 Accuracy:78.9%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_uq_only_for_20.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 2 Accuracy:50.0%
GREEN: Count: 18 Accuracy:83.3%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_uq_only_for_20.png


### Save golden baseline report

In [11]:
cnn_baseline = {
    'model': 'Simple CNN',
    'accuracy': round(cnn_accuracy, 4),
    'classification_report': cnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'flagging_mc_full': mc_full_result,
    'flagging_de_full': de_full_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
}

report_output = generate_report(config,'CIFAR-10 - CNN golden baseline', stage2_result = cnn_baseline )
save_report(report_output, 'TC_10_Simple_CNN_Golden_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
